# General Usage

#### Import pygcc

In [1]:
import pygcc
print(pygcc.__version__)
from pygcc.pygcc_utils import *

1.0.4


#### Read database by specifying the direct- or sequential- access and source database

##### Using the default sequential-access database - speq21.dat

In [3]:
ps = db_reader(sourcedb = './database/thermo.2021.dat', sourceformat = 'gwb')
# ps.dbaccessdic, ps.sourcedic,  ps.specielist

##### Using the user-specified sequential-access database

In [4]:
ps = db_reader(dbaccess = './database/slop07.dat', sourcedb = './database/thermo.2021.dat', sourceformat = 'gwb')
# ps.dbaccessdic, ps.sourcedic,  ps.specielist

Duplicate found for species "AlOH++" in slop07.dat
Duplicate found for species "MgCO3(aq)" in slop07.dat
Duplicate found for species "CaCO3(aq)" in slop07.dat
Duplicate found for species "SrCO3(aq)" in slop07.dat
Duplicate found for species "BaCO3(aq)" in slop07.dat
Duplicate found for species "BaF+" in slop07.dat
Duplicate found for species "Sr(Succ)(aq)" in slop07.dat
Duplicate found for species "Sc(Glut)+(aq)" in slop07.dat
Duplicate found for species "AMP2-" in slop07.dat
Duplicate found for species "HAMP-" in slop07.dat
Duplicate found for species "+H2AMP-" in slop07.dat


### Example: Calculate water properties

*With IAPWS95*

In [26]:
water = iapws95(T = np.array([  0.01, 25, 60,  100, 150,  200,  250,  300]), P = 200)
print(water.rho)
print(water.G)
print(water.H)
print(water.S)

[1009.7358093  1005.83998998  991.70588249  967.43838404  927.6905422
  877.9652085   816.08878794  734.71208434]
[-56194.26867325 -56592.33889827 -57211.87993473 -58000.04190135
 -59093.00868807 -60294.40876238 -61596.26593904 -62995.11966425]
[-68680.33499737 -68236.29564263 -67613.18053974 -66897.34619207
 -65991.92792715 -65062.66635139 -64087.83220993 -63021.29280088]
[15.14005087 16.69550855 18.67153539 20.7005717  22.97720284 25.05223509
 27.00976077 28.9549847 ]


*With ZhangDuan*

In [27]:
water = ZhangDuan(T= np.array([1000, 1050]), P = np.array([1000, 2000]))
print(water.rho, water.G)

[175.8286937  314.74510733] [-91457.05964074 -92350.23024927]


### Example: Calculate water dielectric constants

In [28]:
dielect = water_dielec(T= np.array([1000, 1050]), P = np.array([1000, 2000]), Dielec_method = 'DEW')
dielect.E, dielect.rhohat, dielect.Ah, dielect.Bh

(array([1.19637079, 2.52660344]),
 array([0.17582869, 0.31474511]),
 array([12.8720893 ,  5.29642074]),
 array([0.5403405 , 0.48797969]))

In [29]:
dielect = water_dielec(T= np.array([100, 150]), P = np.array([100, 200]), Dielec_method = 'JN91')
dielect.E, dielect.rhohat, dielect.Ah, dielect.Bh

(array([55.83017195, 44.79388023]),
 array([0.96293375, 0.92769054]),
 array([0.59551378, 0.67352582]),
 array([0.34191435, 0.35183609]))

### Example: Calculate olivine solid solutions

In [30]:
calc = calcRxnlogK( X = 0.85,T = np.array([300, 400, 450]), P = np.array([200, 200, 200]),
                   Specie = 'olivine', dbaccessdic = ps.dbaccessdic, densityextrap = True)
calc.logK, calc.Rxn

(array([ 9.1687495 , 21.15012305, 17.0669597 ]),
 {'type': 'ol',
  'name': 'Fo85',
  'formula': 'Mg1.70Fe0.30Si1O4',
  'MW': 150.1557,
  'min': ['Mg1.70Fe0.30Si1O4',
   ' R&H95, Stef2001',
   -466862.12264868076,
   nan,
   26.21037219328013,
   44.049,
   24.058078393881452,
   0.017393236137667304,
   -890893.8814531548,
   171.38145315487571,
   -3.6586998087954105e-06],
  'spec': ['H+', 'Mg++', 'Fe++', 'SiO2(aq)', 'H2O'],
  'coeff': [-4, 1.7, 0.30000000000000004, 1, 2],
  'nSpec': 5,
  'V': 44.049,
  'source': ' R&H95, Stef2001',
  'elements': ['1.7000', 'Mg', '0.3000', 'Fe', '1.0000', 'Si', '4.0000', 'O']})

### Example: Create new reactions and calculate equilibrium constants

#### An example with pyrite, pyrrhotite, magnetite (PPM) reaction
Pyrite + 4 H<sub>2</sub>O + 2 Pyrrhotite &rarr; 4 H<sub>2</sub>S<sub>(aq)</sub> + Magnetite

Then we can include the reaction in sourcedic, one of the output of db_reader, which is a dictionary of list of reaction coefficients and species. An example with the format for sourcedic is as follows:

> ps.sourcedic['Name'] = ['formula', number of reactants in the reaction, 'coefficient of specie 1', 'specie 1', 'coefficient of specie 2', 'specie 2']

> AB &rarr; 0.5 A<sub>2</sub><sub>(aq)</sub> + B

> ps.sourcedic['AB'] = ['AB', 2, '0.5', 'A2(aq), '1', 'B']

In [31]:
ps.sourcedic['Pyrite'] = ['', 4, '4', 'H2S(aq)', '1', 'Magnetite',  '-2', 'Pyrrhotite', '-4', 'H2O']
Temp = np.array([300.0000, 325, 350.0000, 400.0000, 415, 425.0000, 435, 450.0000])
Press = 500*np.ones(np.size(Temp))
log_K_PPM = calcRxnlogK( T = Temp, P = Press, Specie = 'Pyrite', dbaccessdic = ps.dbaccessdic,
                        sourcedic = ps.sourcedic, specielist = ps.specielist).logK
log_K_PPM

C:\ProgramData\Anaconda3\lib\site-packages\scipy\optimize\minpack.py:175: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  warnings.warn(msg, RuntimeWarning)


array([-9.61842043, -8.38750599, -6.29123074, -4.94288369, -4.26348089,
       -3.81869681, -3.38030713, -2.73438817])

### Example: Generate GWB thermodynamic database

In [9]:
# Vectors for Temperature (C) and Pressure (bar) inputs
T = np.array([  0.010,   25.0000 ,  60.0000,  100.0000, 120.0000,  150.0000,  250.0000,  300.0000])
P = 350*np.ones(np.size(T))
nCa = 1

write GWB using default sourced database, with inclusion of solid_solution and clay thermo properties

In [10]:
%%time
write_database(T = T, P = P, cpx_Ca = nCa, solid_solution = 'Yes',  clay_thermo = 'Yes', 
               dataset = 'GWB')

C:\ProgramData\Anaconda3\lib\site-packages\pygcc\pygcc_utils.py:2266: UserWarning: Some temperature and pressure points are out of aqueous species HKF eqns regions of applicability, hence, density extrapolation has been applied
  warnings.warn('Some temperature and pressure points are out of aqueous species HKF eqns regions of applicability, hence, density extrapolation has been applied')


Success, your new GWB database is ready for download
Wall time: 9.04 s


write GWB using user-specified sourced database, with inclusion of solid_solution and clay thermo properties

In [11]:
%%time
write_database(T = T, P = 175, cpx_Ca = nCa, solid_solution = 'Yes', clay_thermo = 'Yes',
                sourcedb = './database/thermo.29Sep15.dat', dataset = 'GWB')

Success, your new GWB database is ready for download
Wall time: 7.27 s


write GWB using Jan2020 formatted sourced database

In [12]:
%%time
write_database(T = T, P = 125, cpx_Ca = nCa, solid_solution = True, clay_thermo = True,
                sourcedb = './database/thermo.com.tdat', dataset = 'GWB')

Success, your new GWB database is ready for download
Wall time: 8.11 s


write GWB using Jan2020 formatted sourced database with logK as polynomial coefficients, using Tmax and Tmin

In [13]:
%%time
#write_database(T = [0, 450], P = 200, cpx_Ca = nCa, solid_solution = 'Yes', clay_thermo = 'Yes',
#                logK_form = 'polycoeffs', sourcedb =  './database/thermo.com.tdat', dataset = 'GWB')

Success, your new GWB database is ready for download
Wall time: 3min 7s


write GWB using user-specified sourced database and direct-access database (slop07) and FGL97 dielectric constant

In [3]:
%%time
write_database(T = [0, 400], P = 300, cpx_Ca = 0.5, solid_solution = 'Yes', Dielec_method = 'FGL97',
                dbaccess = './database/slop07.dat',
                sourcedb = './database/thermo.29Sep15.dat', dataset = 'GWB')

Duplicate found for species "AlOH++" in slop07.dat
Duplicate found for species "MgCO3(aq)" in slop07.dat
Duplicate found for species "CaCO3(aq)" in slop07.dat
Duplicate found for species "SrCO3(aq)" in slop07.dat
Duplicate found for species "BaCO3(aq)" in slop07.dat
Duplicate found for species "BaF+" in slop07.dat
Duplicate found for species "Sr(Succ)(aq)" in slop07.dat
Duplicate found for species "Sc(Glut)+(aq)" in slop07.dat
Duplicate found for species "AMP2-" in slop07.dat
Duplicate found for species "HAMP-" in slop07.dat
Duplicate found for species "+H2AMP-" in slop07.dat


C:\ProgramData\Anaconda3\lib\site-packages\pygcc\pygcc_utils.py:2266: UserWarning: Some temperature and pressure points are out of aqueous species HKF eqns regions of applicability, hence, density extrapolation has been applied
  warnings.warn('Some temperature and pressure points are out of aqueous species HKF eqns regions of applicability, hence, density extrapolation has been applied')


Success, your new GWB database is ready for download
Wall time: 7.53 s


write GWB using default sourced database and direct-access database (slop07 with Berman mineral data) and FGL97 dielectric constant

In [17]:
%%time
write_database(T = [0, 340], P = 150, Dielec_method = 'FGL97',  dbaccess = './database/slop07.dat',
                dbBerman_dir = './database/berman.dat', dataset = 'GWB',
                mineral_eos = 'Berman88')

Duplicate found for species "AlOH++" in slop07.dat
Duplicate found for species "MgCO3(aq)" in slop07.dat
Duplicate found for species "CaCO3(aq)" in slop07.dat
Duplicate found for species "SrCO3(aq)" in slop07.dat
Duplicate found for species "BaCO3(aq)" in slop07.dat
Duplicate found for species "BaF+" in slop07.dat
Duplicate found for species "Sr(Succ)(aq)" in slop07.dat
Duplicate found for species "Sc(Glut)+(aq)" in slop07.dat
Duplicate found for species "AMP2-" in slop07.dat
Duplicate found for species "HAMP-" in slop07.dat
Duplicate found for species "+H2AMP-" in slop07.dat
Duplicate found for species "Cordierite" in berman.dat
Success, your new GWB database is ready for download
Wall time: 4.5 s


write GWB using user-specified sourced Pitzer database and default direct-access database along the saturation curve

In [16]:
%%time
write_database(T = [0, 350], P = 'T', dataset = 'GWB',  sourcedb = './database/thermo_hmw.tdat')

Success, your new GWB database is ready for download
Wall time: 1.53 s


write GWB using user-specified sourced EQ3/6 Pitzer database and default direct-access database

In [38]:
%%time
write_database(T = [0, 350], P = 250, dataset = 'GWB', sourcedb = './database/data0.hmw',
               sourceformat = 'EQ36')

Success, your new GWB database is ready for download
Wall time: 2.77 s


### Example: Generate EQ3/6 thermodynamic database

write EQ3/6 using default sourced database

In [18]:
%%time
write_database(T = T, P = P, cpx_Ca = 1, solid_solution = 'Yes', clay_thermo = 'Yes', 
               dataset = 'EQ36')

C:\ProgramData\Anaconda3\lib\site-packages\pygcc\pygcc_utils.py:3327: UserWarning: Some temperature and pressure points are out of aqueous species HKF eqns regions of applicability, hence, density extrapolation has been applied
  warnings.warn('Some temperature and pressure points are out of aqueous species HKF eqns regions of applicability, hence, density extrapolation has been applied')


Success, your new EQ3/6 database is ready for download
Wall time: 9.14 s


write EQ3/6 user-specified sourced database

In [37]:
%%time
write_database(T = [0, 400], P = 350, cpx_Ca = 0.5, solid_solution = 'Yes', clay_thermo = 'Yes',
                sourcedb = './database/data0.geo', dataset = 'EQ36', sourcedb_codecs = 'latin-1')

Success, your new EQ3/6 database is ready for download
Wall time: 8.9 s


write EQ3/6 using user-specified sourced Pitzer database

In [20]:
%%time
write_database(T = [0, 350], P = 200, sourcedb = './database/data0.hmw', dataset = 'EQ36')

Success, your new EQ3/6 database is ready for download
Wall time: 1.66 s


write EQ3/6 user-specified sourced database using FGL97 dielectric constant

In [9]:
%%time
write_database(T = [0, 400], P = 300, cpx_Ca = 0.1, sourcedb = './database/data0.geo', 
               dataset = 'EQ36', solid_solution = 'Yes', Dielec_method = 'FGL97', clay_thermo = 'Yes')

Success, your new EQ3/6 database is ready for download
Wall time: 9.4 s


write EQ3/6 user-specified sourced database using DEW model

In [29]:
%%time
Temp = np.array([50, 100, 150, 300, 450, 500, 600, 700])
write_database(T = Temp, P = 1500, sourcedb = './database/data0.geo', dataset = 'EQ36', 
               Dielec_method = 'DEW')

Success, your new EQ3/6 database is ready for download
Wall time: 4.88 s


### Example: Generate ToughReact thermodynamic database

write ToughReact using user-specified EQ3/6 database

In [24]:
%%time
write_database(T = T, P = P, cpx_Ca = nCa, solid_solution = 'Yes', sourcedb = './database/data0.dat',
                dataset = 'ToughReact', sourceformat = 'EQ36')

C:\ProgramData\Anaconda3\lib\site-packages\pygcc\pygcc_utils.py:4365: UserWarning: Some temperature and pressure points are out of aqueous species HKF eqns regions of applicability, hence, density extrapolation has been applied
  warnings.warn('Some temperature and pressure points are out of aqueous species HKF eqns regions of applicability, hence, density extrapolation has been applied')


Success, your new ToughReact database is ready for download
Wall time: 9.2 s


write ToughReact using user-specified GWB database

In [15]:
%%time
write_database(T = [0, 350], P = 250, cpx_Ca = 0.25, clay_thermo = 'Yes', dataset = 'ToughReact',
                sourcedb = './database/thermo.com.tdat', sourceformat = 'GWB')

Success, your new ToughReact database is ready for download
Wall time: 4.33 s


write ToughReact using EQ3/6 user-specified Ptizer sourced database and JN91 dielectric constant

In [11]:
%%time
write_database(T = [0, 300], P = 200, sourceformat = 'EQ36', sourcedb = './database/data0.hmw',
                dataset = 'ToughReact', Dielec_method = 'JN91')

Success, your new ToughReact database is ready for download
Wall time: 1.77 s


### Example: Generate Pflotran thermodynamic database

 write Pflotran using user-specified EQ3/6 database

In [30]:
%%time
write_database(T = T, P = P, clay_thermo = 'Yes', sourcedb = './database/data0.dat',
                dataset = 'Pflotran', sourceformat = 'EQ36')

C:\ProgramData\Anaconda3\lib\site-packages\pygcc\pygcc_utils.py:3938: UserWarning: Some temperature and pressure points are out of aqueous species HKF eqns regions of applicability, hence, density extrapolation has been applied
  warnings.warn('Some temperature and pressure points are out of aqueous species HKF eqns regions of applicability, hence, density extrapolation has been applied')


Success, your new Pflotran database is ready for download
Wall time: 5.47 s


write Pflotran using user-specified GWB database

In [13]:
%%time
write_database(T = [0, 350], P = 250, cpx_Ca = 0.1, solid_solution = True, clay_thermo = True,
               sourcedb = './database/thermo.com.tdat', dataset = 'Pflotran', sourceformat = 'GWB')

C:\ProgramData\Anaconda3\lib\site-packages\pygcc\pygcc_utils.py:3938: UserWarning: Some temperature and pressure points are out of aqueous species HKF eqns regions of applicability, hence, density extrapolation has been applied
  warnings.warn('Some temperature and pressure points are out of aqueous species HKF eqns regions of applicability, hence, density extrapolation has been applied')


Success, your new Pflotran database is ready for download
Wall time: 7.56 s


### Example: Calculate clay mineral thermodynamics

In [32]:
#%% specify the direct access thermodynamic database
db_dic = db_reader(dbaccess = './database/speq21.dat').dbaccessdic

folder_to_save = 'output'
if os.path.exists(os.path.join(os.getcwd(), folder_to_save)) == False:
    os.makedirs(os.path.join(os.getcwd(), folder_to_save)) 
fid = open('./output/logK_05.txt', 'w')

logKRxn = calcRxnlogK(T = T, P = P, Specie = 'Clay', dbaccessdic = db_dic,
                        elem = ['Clinochlore', '3', '2', '0', '0', '5', '0', '0', '0', '0'],
                        densityextrap = True,  group = '14A')
logK, Rxn = logKRxn.logK, logKRxn.Rxn

# output in EQ36 format
outputfmt(fid, logK, Rxn, dataset = 'EQ36')
# output in GWB format
outputfmt(fid, logK, Rxn, dataset = 'GWB')
# output in Pflotran format
outputfmt(fid, logK, Rxn, dataset = 'Pflotran')
# output in ToughReact format
outputfmt(fid, logK, Rxn, dataset = 'ToughReact')
fid.close()

### Example: Calculate plagioclase solid-solution thermodynamics

In [33]:
#%% specify the direct access thermodynamic database
db_dic = db_reader(dbaccess = './database/speq21.dat').dbaccessdic

folder_to_save = 'output'
if os.path.exists(os.path.join(os.getcwd(), folder_to_save)) == False:
    os.makedirs(os.path.join(os.getcwd(), folder_to_save)) 
fid = open('./output/logK_05.txt', 'w')

logKRxn = calcRxnlogK(T = T, dbaccessdic = db_dic, P = 'T', X = 0.634, Specie = 'Plagioclase',
                        densityextrap = True)
logK, Rxn = logKRxn.logK, logKRxn.Rxn

# output in EQ36 format
outputfmt(fid, logK, Rxn, dataset = 'EQ36')
# output in GWB format
outputfmt(fid, logK, Rxn, dataset = 'GWB')
# output in Pflotran format
outputfmt(fid, logK, Rxn, dataset = 'Pflotran')
# output in ToughReact format
outputfmt(fid, logK, Rxn, dataset = 'ToughReact')
fid.close()

### Example: Calculate CO$_2$ activity and molality

In [34]:
T = np.array([  0.010,   25 ,  60,  100, 150,  175,  200,  250])
P = 250*np.ones(np.size(T))

TK = convert_temperature(T, Out_Unit = 'K')

#%% Calculate CO2 activity and molality at ionic strength of 0.5M
# with Duan_Sun
log10_co2_gamma, mco2 = Henry_duan_sun(TK, P, 0.5)
co2_activity = 10**log10_co2_gamma

# with Drummond
log10_co2_gamma = drummondgamma(TK, 0.5)
co2_activityD = 10**log10_co2_gamma

In [35]:
for i in range(len(T)):
    print('Fluid Temperature [C]: ', T[i])
    print('Fluid Pressure [bar]: ', P[i])
    print('Molality of CO2 in aqueous phase: ', mco2.ravel()[i])
    print('Activity of CO2 in aqueous phase (Duan_Sun): ', co2_activity.ravel()[i])
    print('Activity of CO2 in aqueous phase (Drummond): ', co2_activityD.ravel()[i])
    print('\n')

Fluid Temperature [C]:  0.01
Fluid Pressure [bar]:  250.0
Molality of CO2 in aqueous phase:  3.231443164427958
Activity of CO2 in aqueous phase (Duan_Sun):  1.1291811186477334
Activity of CO2 in aqueous phase (Drummond):  1.1340282640704162


Fluid Temperature [C]:  25.0
Fluid Pressure [bar]:  250.0
Molality of CO2 in aqueous phase:  2.089130347640426
Activity of CO2 in aqueous phase (Duan_Sun):  1.117606348338049
Activity of CO2 in aqueous phase (Drummond):  1.1228777554452154


Fluid Temperature [C]:  60.0
Fluid Pressure [bar]:  250.0
Molality of CO2 in aqueous phase:  1.4387910250773501
Activity of CO2 in aqueous phase (Duan_Sun):  1.1097025985737983
Activity of CO2 in aqueous phase (Drummond):  1.1184645553679928


Fluid Temperature [C]:  100.0
Fluid Pressure [bar]:  250.0
Molality of CO2 in aqueous phase:  1.1971664693082034
Activity of CO2 in aqueous phase (Duan_Sun):  1.109458812741491
Activity of CO2 in aqueous phase (Drummond):  1.1250331593722136


Fluid Temperature [C]:  150

### Example: Calculate water activity

In [36]:
#%% Calculate Water activity, osmotic coefficient and NaCl mean activity coefficient at an ionic strength of 0.5M
aw, phi, mean_act = Helgeson_activity(T, P, 0.5, Dielec_method = 'JN91')
for i in range(len(T)):
    print('Fluid Temperature [C]: ', T[i])
    print('Fluid Pressure [bar]: ', P[i])
    print('Water activity: ', aw.ravel()[i])
    print('Water osmotic coefficient: ', phi.ravel()[i])
    print('NaCl mean activity coefficient: ', mean_act.ravel()[i])
    print('\n')

Fluid Temperature [C]:  0.01
Fluid Pressure [bar]:  250.0
Water activity:  0.983412433698784
Water osmotic coefficient:  0.9284718584433971
NaCl mean activity coefficient:  0.6926294758605518


Fluid Temperature [C]:  25.0
Fluid Pressure [bar]:  250.0
Water activity:  0.9834694019509086
Water osmotic coefficient:  0.9252563947920361
NaCl mean activity coefficient:  0.6830156908317317


Fluid Temperature [C]:  60.0
Fluid Pressure [bar]:  250.0
Water activity:  0.9834545247516545
Water osmotic coefficient:  0.9260960917803105
NaCl mean activity coefficient:  0.674127423413536


Fluid Temperature [C]:  100.0
Fluid Pressure [bar]:  250.0
Water activity:  0.9836228186833588
Water osmotic coefficient:  0.9165980079014355
NaCl mean activity coefficient:  0.6470880031602849


Fluid Temperature [C]:  150.0
Fluid Pressure [bar]:  250.0
Water activity:  0.9839613772269228
Water osmotic coefficient:  0.8974955418879642
NaCl mean activity coefficient:  0.6014993581954272


Fluid Temperature [C]:  1